In [133]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [134]:
# base imports
import numpy as np 
import pandas as pd
from functools import partial
import pickle
import warnings

# preprocessing
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from bs4 import BeautifulSoup

import re

# NLP packages
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
import scipy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


max_length = 32

size = 250

In [135]:

"""
Data Preprocessing Functions
"""
stemmer = PorterStemmer()
lemma = WordNetLemmatizer()

def tokenize(text,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    tokens = [word for word in text.lower() if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # -1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens


def token2index(tokens,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def assert_even_length_array(arr):
    if len(arr) % 2 != 0:
        raise AssertionError("Array does not have an even number of elements: {}".format(len(arr)))

def check_order(array):
    prev_entry = None
    ilist = []  
    for i, entry in enumerate(array):
        if entry == prev_entry:
            ilist.append(i)
    
        prev_entry = entry

    return ilist

def sort_data_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array.iloc[i] = temp2
        array.iloc[i+1] = temp1

    return array

def sort_label_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array[i] = temp2
        array[i+1] = temp1

    return array


def get_vector(vec1,vec2):
    
    d_vec = vec2 - vec1
    
    if np.sum(d_vec) == 0:
        
        warnings.warn("A text vector and its' counterfactual vector have the same value. This is probably not right.")
        
    if isinstance(d_vec, scipy.sparse.csr_matrix):  # or any other sparse matrix type
        mag = scipy.linalg.norm(d_vec.todense())

    elif isinstance(d_vec, np.ndarray):
        mag = np.linalg.norm(d_vec)
        # Perform operations specific to dense arrays
    else:
        print("Unknown vector type")
        mag = np.linalg.norm(d_vec)
    
    if mag!=0:
        return d_vec/mag        
    else:
        return np.zeros((np.shape(vec1)[0]))
    # return [d_vec/mag]



## cleaning the text

def cleantext(text):
    # removing the "\"
    text = re.sub("'\''","",text)
    
    # removing special symbols
    text = re.sub("[^ a-zA-Z]","",text)
    
    # removing the whitespaces
    text = ' '.join(text.split())
    
    # convert text to lowercase    
    text = text.lower()
    
    return text

# removing the stopwords
stop_words = set(stopwords.words('english'))
mod_stop_words = set(stopwords.words('english')) - {'not', 'but'}
# stop_words = stop_words - set(['dont','do'])
# def removestopwords(text):
    
#     removedstopword = [word for word in text.split() if word not in mod_stop_words]
#     return ' '.join(removedstopword)


def removestopwords(text):
    
    removedstopword = [word for word in text if word not in mod_stop_words]
    return removedstopword

#Removing the html strips 
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

#Removing the square brackets
def remove_between_square_brackets(text):
    return re.sub(r'\[[^]]*\]', '', text)

#Removing Emails
def remove_Emails(text):
    pattern=r'\S+@\S+'
    text=re.sub(pattern,'',text)
    return text

#Removing URLS
def remove_URLS(text):
    pattern=r'http\S+'
    text=re.sub(pattern,'',text)
    return text

def remove_special_characters(text, remove_digits=True):
    pattern=r'[^a-zA-z0-9\s]'
    text=re.sub(pattern,' ',text)
    return text

#Removing numbers
def remove_numbers(text):
    pattern = r'\d+'
    text = re.sub(pattern, '', text)
    return text

def lematizing(sentence):
    stemSentence = ""
    for word in sentence.split():
        stem = lemma.lemmatize(word)
        stemSentence += stem
        stemSentence += " "
    stemSentence = stemSentence.strip()
    return stemSentence

def stemming(sentence):
    
    stemmed_sentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemmed_sentence+=stem
        stemmed_sentence+=" "
        
    stemmed_sentence = stemmed_sentence.strip()
    return stemmed_sentence



def process_data(data):
    
    data=data.apply(lambda x:strip_html(x))
    data=data.apply(remove_between_square_brackets)
    data = data.apply(lambda x:cleantext(x))
    data=data.apply(remove_special_characters)
    data=data.apply(remove_URLS)
    data=data.apply(remove_Emails)
    data=data.apply(remove_numbers)


    data = data.apply(lambda x: word_tokenize(x.lower()))

    # Step 3: remove stopwords and stem (token level)
    """Uncomment two below for OG dataset"""
    data = data.apply(lambda tokens: [stemming(token) for token in removestopwords(tokens)])
    data = data.apply(lambda tokens: [stemming(token) for token in tokens])
    
    # data = data.apply(lambda x: word_tokenize(x)[:32])
    # # #lemmatizing
    # data = data.apply(lambda x: lematizing(x))



    # #stemming
    # data = data.apply(lambda text:stemming(text))
    # data = data.apply(lambda x:removestopwords(x))
    # data = [text_to_vector(txt) for txt in data]
    
    return data



def gen_knowledge(data,label,cf=False):
    """
    Takes sequences of pairs of counterfactuals and 
    returns just the originals, with the counterfactual 
    included as an annotation.

    If cf=True, returns the counterfactuals and omits the originals
    """
    
    vectors_in = data['vector']
    text_in = data['text']
    # sparse_array = sp.csr_matrix((0,dlength))

    text_out=[]
    vectors_out = []
    labels = []
    # vectors = np.empty((0,1,dlength))
    cf_text = []
    cf_labels = []
    cf_vectors = []
    
    for i in range(int(np.shape(vectors_in)[0]/2)):
        
        
        if cf:
            vec1,vec2 = vectors_in[i * 2 + 1],vectors_in[i * 2]  # Get the data from the current dataset slice
            label_out = label[i*2+1]
            text = text_in[i * 2 + 1]
            _cf_text=text_in[i * 2]
        else:
            vec1,vec2 = vectors_in[i * 2],vectors_in[i * 2 + 1]  # Get the data from the current dataset slice                
            label_out = label[i*2]
            text = text_in[i * 2]
            _cf_text = text_in[i * 2 + 1]
            
        text_out.append(text)
        labels.append(label_out)
        vectors_out.append(vec1)
    
        cf_vectors.append([vec2])
        cf_labels.append([1 - label_out])
        cf_text.append([_cf_text])
    
    return {'vector':vectors_out,'text':text_out},labels,cf_vectors, cf_labels,cf_text



def compile_K(data,label, paired=False,cf=False,int_out=False):
    
    if paired:
        
        if np.shape(data['vector'])[0]%2 != 0:
            raise ValueError("Can't generate paired counterfactuals with an uneven number of samples")
        
        data,label,vector,labels,cf_text = gen_knowledge(data,label,cf=cf)
        
    else:
        warnings.warn("Non-paired data just creates blanks atm")
        
        vector = [[] for _ in range(np.shape(data['vector'])[0])]
        cf_text = [[] for _ in range(np.shape(data['vector'])[0])]

        labels = [[]]*len(vector)

    magnitude = np.ones(len(vector))
    magnitude = np.expand_dims(magnitude, axis=1)
    

    if int_out:
        for i in range(np.shape(vector)[0]):
            vector[i] = vector[i].astype(int)

    n_samples = len(data['text'])
    print(f'Returning {n_samples} samples with {len(vector)} counterfactuals')
    
    # return {'text':data['text'],
    #         'X':data['vector'],
    #         'Y':label,
    #         'K':{'vector':vector,'label':labels,'magnitude':magnitude, 'text':cf_text}}
    return {'text':list(data['text']),
            'X':data['vector'],
            'Y':list(label),
            'K':{'X':np.expand_dims(data['vector'],axis=1),
                 'Y':np.expand_dims(np.full_like(np.array(label),np.nan),axis=1),
                 'K':vector, #np.expand_dims(vector,axis=1),
                 'magnitude':np.expand_dims(magnitude,axis=1),
                 'text':np.expand_dims(cf_text,axis=1)}}

In [136]:

# this is just the counterfactually augmented dat from kaushik
train_data = pd.read_table("data/paired/train_paired.tsv")
test_data = pd.read_table("data/paired/test_paired.tsv")
dev_data = pd.read_table("data/paired/dev_paired.tsv")
combined_data = pd.concat([train_data, test_data, dev_data], ignore_index=True)

dev_paired = dev_data['Text']
dev_label = np.array(dev_data['Sentiment'].map({'Positive': 1, 'Negative': 0}))

test_paired = test_data['Text']
test_label = np.array(test_data['Sentiment'].map({'Positive': 1, 'Negative': 0}))
 
train_paired = train_data['Text']
train_label = np.array(train_data['Sentiment'].map({'Positive': 1, 'Negative': 0}))


In [137]:

dev_paired_processed = process_data(dev_paired)
test_paired_processed = process_data(test_paired)
train_paired_processed = process_data(train_paired)



In [138]:
train_paired

0       Long, boring, blasphemous. Never have I been s...
1       Long, fascinating, soulful. Never have I been ...
2       Not good! Rent or buy the original! Watch this...
3       So good! Rent or buy the original, too! Watch ...
4       This movie is so bad, it can only be compared ...
                              ...                        
3409    Bacall does badly here - even considering this...
3410    Eddie Murphy plays Chandler Jarrell, a man who...
3411    Eddie Murphy plays Chandler Jarrell, a man who...
3412    ZP is deeply related to that youth dream repre...
3413    ZP is deeply related to that youth dream repre...
Name: Text, Length: 3414, dtype: object

In [139]:
train_paired_processed

0       [long, bore, blasphem, never, glad, see, end, ...
1       [long, fascin, soul, never, sad, see, end, cre...
2       [not, good, rent, buy, origin, watch, someon, ...
3       [good, rent, buy, origin, watch, goodit, amaz,...
4       [movi, bad, compar, alltim, worst, comedi, pol...
                              ...                        
3409    [bacal, badli, even, consid, nd, film, one, of...
3410    [eddi, murphi, play, chandler, jarrel, man, de...
3411    [eddi, murphi, play, chandler, jarrel, man, de...
3412    [zp, deepli, relat, youth, dream, repr, hippi,...
3413    [zp, deepli, relat, youth, dream, repr, hippi,...
Name: Text, Length: 3414, dtype: object

In [140]:

# # Create tokenizer limited to top 20k words
# tokenizer = Tokenizer(num_words=20000, oov_token="<UNK>")
# tokenizer.fit_on_texts(train_paired)

# # Convert sentences to sequences of integers
# tfidf_train = tokenizer.texts_to_sequences(train_paired)
# tfidf_train = pad_sequences(tfidf_train, maxlen=32, padding="post", truncating="post")

# tfidf_test = tokenizer.texts_to_sequences(test_paired)
# tfidf_test = pad_sequences(tfidf_test, maxlen=32, padding="post", truncating="post")

# tfidf_dev = tokenizer.texts_to_sequences(dev_paired)
# tfidf_dev = pad_sequences(tfidf_dev, maxlen=32, padding="post", truncating="post")


In [141]:
data_for_vectorizer = train_paired_processed.apply(lambda tokens: " ".join(tokens))
vectorizer = CountVectorizer(max_features=20000)
vectorizer.fit(data_for_vectorizer)
# Get the vocabulary mapping (word to integer index)
vocabulary = vectorizer.vocabulary_


# define a reusable lambda
to_indices = lambda texts: [token2index(text, vocabulary, max_length=max_length) for text in texts]


# now you can call it on any dataset
tfidf_train = to_indices(train_paired_processed)
tfidf_dev   = to_indices(dev_paired_processed)
tfidf_test  = to_indices(test_paired_processed)


In [142]:

# tfidf_train_cf = to_indices(process_data(cf_train))

In [143]:

"""
########################################################################################################################
Save embeddings
########################################################################################################################
"""
# Stripping html from unprocessed text, just to clean it up
print('\ntrain_Data')
cf_X = {'text':[strip_html(txt) for txt in train_data['Text']],'vector':np.array(tfidf_train)}
cf_train={'original': compile_K(cf_X,train_label,paired=True ),
         'modified': compile_K(cf_X,train_label,paired=True,cf=True)
         }

print('\ndev_Data')
cf_X = {'text':[strip_html(txt) for txt in dev_data['Text']],'vector':np.array(tfidf_dev)}
cf_dev ={'original': compile_K(cf_X,dev_label,paired=True),
         'modified': compile_K(cf_X,dev_label,paired=True,cf=True)
         }

print('\ntest_Data')
cf_X = {'text':[strip_html(txt) for txt in test_data['Text']],'vector':np.array(tfidf_test)}
cf_test={'original': compile_K(cf_X,test_label,paired=True),
         'modified': compile_K(cf_X,test_label,paired=True,cf=True)
         }


pickle_data =   {'train':cf_train,
                'test':cf_test,
                'dev':cf_dev,
                'n_classes':2}

embedding_path = f'data/integer_len{max_length}.pkl'

with open(embedding_path, 'wb') as file:
    pickle.dump(pickle_data, file)


train_Data
Returning 1707 samples with 1707 counterfactuals
Returning 1707 samples with 1707 counterfactuals

dev_Data
Returning 245 samples with 245 counterfactuals
Returning 245 samples with 245 counterfactuals

test_Data
Returning 488 samples with 488 counterfactuals
Returning 488 samples with 488 counterfactuals


/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment-cli/cli_venv/lib/python3.10/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


In [144]:
total = 0
for i, (x, k) in enumerate(list(zip(cf_train['original']['K']['X'],cf_train['original']['K']['K']))):
    # print(x[0])
    # print(k)
    if (x[0] == k[0]).all():
        total +=1

        # print(cf_train['original']['K']['text'][i])
        # print(cf_train['original']['text'][i])

print(total)

151


In [145]:
i

1706

In [146]:
cf_train['original']['K']['text']

array([[['Long, fascinating, soulful. Never have I been so sad to see ending credits roll.']],

       [["So good! Rent or buy the original, too! Watch this, too! It's just as good!It is an amazing Elvis impersonator and the real King."]],

       [['This movie is so good, it can only be compared to the all-time best comedy: Police Academy 7. Great laughs throughout the movie. Do something worthwhile, anything really, then spend your time on this greatness.']],

       ...,

       [['Bacall does badly here - even considering this is only her 2nd film. This one is often overshadowed because it falls between 2 great successes: "To Have and To Have Not" (1944) and "The Big Sleep" (1945), both of which paired her with Humphrey Bogart. Granted this one is not up to par to the other movies but I think through no fault of her own. I think there was some miscasting in having her portray a British upper-crust lady. No accent whatsoever. I think all the strange accents were distracting - Boyer 

In [147]:
print(cf_train['original']['text'][0])
print(train_paired_processed[0])
print(tfidf_train[0])
print(cf_train['modified']['text'][0])
print(train_paired_processed[1])
print(tfidf_train[1])


Long, boring, blasphemous. Never have I been so glad to see ending credits roll.
['long', 'bore', 'blasphem', 'never', 'glad', 'see', 'end', 'credit', 'roll']
[np.int64(8905), np.int64(1693), np.int64(1533), np.int64(10366), np.int64(6182), np.int64(13396), np.int64(4694), np.int64(3276), np.int64(12913), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Long, fascinating, soulful. Never have I been so sad to see ending credits roll.
['long', 'fascin', 'soul', 'never', 'sad', 'see', 'end', 'credit', 'roll']
[np.int64(8905), np.int64(5268), np.int64(14203), np.int64(10366), np.int64(13071), np.int64(13396), np.int64(4694), np.int64(3276), np.int64(12913), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
